# Imports

In [20]:
from dotenv import load_dotenv
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
from autogen_core.models import UserMessage
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console
from datetime import datetime
from typing import List, Dict
import asyncio

In [2]:
load_dotenv()

True

# Model setup 

In [38]:
model_client = AzureOpenAIChatCompletionClient(
    azure_deployment="gpt-5-mini",      # Deployment name in Azure
    model="gpt-5-mini",                 # Actual model behind deployment
    api_version="2024-12-01-preview",
    azure_endpoint="https://openai-api-management-gw.azure-api.net/",
)


In [ ]:
# response = await model_client.create([
#         UserMessage(content="Explain autogen in one line", source="user")
#     ])

In [ ]:
# response

CreateResult(finish_reason='stop', content='Autogen automatically generates code, configurations, or other content from templates, specifications, or models to speed and standardize development.', usage=RequestUsage(prompt_tokens=13, completion_tokens=291), cached=False, logprobs=None, thought=None)

# Sample documents for demonstration

In [12]:
SAMPLE_DOCUMENTS = [
    {
        "id": "DOC001",
        "title": "Employee Remote Work Policy",
        "type": "HR Policy",
        "content": """
        Remote Work Policy
        Effective Date: January 2023

        1. Eligibility: All full-time employees
        2. Equipment: Company will provide laptop
        3. Work Hours: Standard 9-5 EST

        Note: This policy supersedes the 2021 remote work guidelines.
        """,
        "last_updated": "2023-01-15"
    },
    {
        "id": "DOC002",
        "title": "Data Security Manual",
        "type": "Technical Manual",
        "content": """
        Data Security Manual
        Version: 2.1

        1. Password Requirements: Minimum 8 characters
        2. Encryption: Use AES-256 for sensitive data

        Missing sections:
        - Incident Response Procedures
        - Data Retention Policy
        """,
        "last_updated": "2024-03-20"
    },
    {
        "id": "DOC003",
        "title": "Customer Service Guidelines",
        "type": "Operational Guidelines",
        "content": """
        Customer Service Guidelines

        1. Response Time: Within 24 hours
        2. Escalation: After 48 hours to supervisor
        3. Communication: Use template responses from 2019 handbook

        Approved by: [Missing Signature]
        Review Date: [Not Specified]
        """,
        "last_updated": "2024-11-10"
    }
]

# Define specialized agents

In [ ]:
class DocumentComplianceSystem:
    def __init__(self):
        self.document_scanner = AssistantAgent(
            name="DocumentScanner",
            model_client=model_client,
            system_message="""You are a Document Scanner agent responsible for:
            1. Monitoring document repositories for new or updated documents
            2. Extracting document metadata (title, type, last updated date)
            3. Preparing documents for compliance analysis
            4. Identifying document categories (HR, Technical, Operational)

            Format your findings clearly with document ID, type, and key metadata."""
        )

        self.compliance_checker = AssistantAgent(
            name="ComplianceChecker",
            model_client=model_client,
            system_message="""You are a Compliance Checker agent responsible for:
            1. Analyzing documents for compliance issues such as:
               - Missing required sections (approvals, dates, signatures)
               - Outdated references or superseded content
               - Inconsistent terminology or formatting
               - Version control issues
            2. Categorizing issues by severity (Critical, Major, Minor)
            3. Providing specific recommendations for remediation

            Always structure your analysis with: Issue Type | Severity | Details | Recommendation"""
        )

        self.review_coordinator = AssistantAgent(
            name="ReviewCoordinator",
            model_client=model_client,
            system_message="""You are a Review Coordinator agent responsible for:
            1. Processing compliance check results
            2. Determining routing based on severity:
               - Critical: Immediate escalation to Legal/Executive review
               - Major: Route to department head for approval
               - Minor: Send revision request to document owner
            3. Generating action items with deadlines
            4. Creating audit trail for compliance tracking

            Format routing decisions with: Document | Issues | Routing Decision | Action Items | Due Date
            
            Important rules:
            - You are the final agent.
            - Do not ask another agent to continue.
            - End your final response with exactly: TERMINATE
            """
            
        )

        termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(10)

        self.team = SelectorGroupChat(
            participants=[
                self.document_scanner,
                self.compliance_checker,
                self.review_coordinator,
            ],
            model_client=model_client,
            termination_condition=termination,
            allow_repeated_speaker=False,
        )

    
    # def generate_compliance_report(self):
    #     print("\n" + "=" * 60)
    #     print("COMPLIANCE WORKFLOW SUMMARY")
    #     print("=" * 60)
    #     print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    #     print(f"Documents Processed: {len(SAMPLE_DOCUMENTS)}")
    #     print("\nKey Findings:")
    #     print("- Missing required sections detected in technical documentation")
    #     print("- Outdated references found in operational guidelines")
    #     print("- Signature approvals missing in customer service documents")
    #     print("\nWorkflow Demonstration Complete")
        
    async def process_document_batch(self, documents:List[Dict]):
        print("Starting AutoGen Document Compliance Workflow\n")
        print("=" * 60)

        doc_summary = "\n\n".join([
            f"Document ID: {doc['id']}\n"
            f"Title: {doc['title']}\n"
            f"Type: {doc['type']}\n"
            f"Last Updated: {doc['last_updated']}\n"
            f"Content Preview:\n{doc['content']}"
            for doc in documents
        ])

        initial_message = f"""
        New documents have been detected in the repository. Please process these documents through our compliance workflow:

        {doc_summary}

        Workflow Steps:
        1. DocumentScanner: Analyze and categorize the documents
        2. ComplianceChecker: Perform compliance analysis on each document
        3. ReviewCoordinator: Determine routing and create action items

        Begin the analysis.

        After ReviewCoordinator finishes, end with conversation.
        """

        stream = self.team.run_stream(
            task=initial_message,
        )

    
        async for message in stream:
            if hasattr(message, "source"):
                print(f"\n{'='*60}")
                print(f"{message.source}")
                print(f"{'='*60}")
                print(message.content)

        # await Console(stream)

        # self.generate_compliance_report()
        

In [42]:
async def run_demo():
    print("AutoGen Document Compliance System Demo")
    print("=" * 60)
    print("Demonstrating key AutoGen features:")
    print("✓ Multi-agent collaboration")
    print("✓ Declarative agent configuration style")
    print("✓ Group chat orchestration")
    print("✓ Automated workflow execution")
    print("=" * 60 + "\n")

    compliance_system = DocumentComplianceSystem()
    await compliance_system.process_document_batch(SAMPLE_DOCUMENTS)

    await model_client.close()



In [43]:
await run_demo()

AutoGen Document Compliance System Demo
Demonstrating key AutoGen features:
✓ Multi-agent collaboration
✓ Declarative agent configuration style
✓ Group chat orchestration
✓ Automated workflow execution

Starting AutoGen Document Compliance Workflow


user

        New documents have been detected in the repository. Please process these documents through our compliance workflow:

        Document ID: DOC001
Title: Employee Remote Work Policy
Type: HR Policy
Last Updated: 2023-01-15
Content Preview:

        Remote Work Policy
        Effective Date: January 2023

        1. Eligibility: All full-time employees
        2. Equipment: Company will provide laptop
        3. Work Hours: Standard 9-5 EST

        Note: This policy supersedes the 2021 remote work guidelines.
        

Document ID: DOC002
Title: Data Security Manual
Type: Technical Manual
Last Updated: 2024-03-20
Content Preview:

        Data Security Manual
        Version: 2.1

        1. Password Requirements: Minimum 8 cha